# PGC phase measurements: LVM analysis

This notebook analyses the LabVIEW `.lvm` files recorded with the PGC (phase-generated carrier) program.
It's a Python port of `LVM_reader.m`.

1. **Single file**: loads every segment, shows the phase time traces and the power spectral density (PSD).
2. **Multiple files**: draws a chronogram of all files, the PSD of the phase after a slow low-pass trend is removed, plus each file's mean phase and minimum block standard deviation.

All plots are **interactive**. Drag to zoom (on an axis to zoom only that axis), double-click to reset, and click a legend entry to hide a trace.

The parsing and signal-processing code is in [`lvm_reader.py`](lvm_reader.py).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

sys.path.insert(0, str(Path.cwd().parent))   # make lvm_reader importable from notebooks/
import lvm_reader as lvm

# Interactive in Jupyter and in the exported HTML (plotly.js loaded from CDN).
# Only the HTML form is stored, so each plot is saved once in the .ipynb.
pio.renderers.default = "notebook_connected"

# Categorical colours, fixed order: blue, orange, aqua, yellow, magenta, green, violet, red
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
pio.templates["lab"] = go.layout.Template(layout=dict(
    colorway=SERIES, font=dict(family="system-ui, sans-serif", size=12, color="#52514e"),
    paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb", height=450,
    margin=dict(l=70, r=20, t=50, b=50), hovermode="x unified",
    title=dict(x=0, xanchor="left", font=dict(color="#0b0b0b", size=15)),
    xaxis=dict(gridcolor="#e4e3de", linecolor="#b5b4ad", zeroline=False, showspikes=True,
               spikemode="across", spikethickness=1, spikecolor="#b5b4ad"),
    yaxis=dict(gridcolor="#e4e3de", linecolor="#b5b4ad", zeroline=False),
    legend=dict(bgcolor="rgba(0,0,0,0)"),
))
pio.templates.default = "lab"
PLOT_CONFIG = dict(scrollZoom=True, displaylogo=False)

## Configuration

Set `DATA_DIR` to the folder that holds the measurements. Extensions don't matter: some files were saved without `.lvm`, so a file is recognised as LVM by its `LabVIEW Measurement` signature.

`MAX_POINTS` limits the points stored per trace. A one-hour recording has 3.6 million samples, too many to embed in a web page, so traces are min/max decimated. Spikes and the envelope are preserved, but detail finer than about `samples / MAX_POINTS` samples isn't visible when you zoom. To inspect a short interval at full resolution, use `T_WINDOW`.

In [2]:
DATA_DIR = Path("../05luglio")

SINGLE_FILE = "AB.lvm"
T_WINDOW    = None        # e.g. (100, 110): plot only this time span (s) at full resolution

# Files for the multi-file comparison, in chronological order
CHRONO_POINTS = 30_000    # points per file in the multi-file chronogram
MULTI_FILES = ["baselineDIW_1", "baselinePBS_1", "PEG_1", "AB.lvm", "spike01.lvm", "spike1.lvm", "spike10.lvm"]
LABELS      = ["DIW", "PBS", "PEG", "AB", "0.1 nM", "1 nM", "10 nM"]

PHASE_CH     = "Phase1"   # channel used in the multi-file analysis
UNWRAP       = False      # phase is already unwrapped by LabVIEW (values span >1e5 degrees)
TREND_CUTOFF = 1e-4       # low-pass cutoff normalised to Nyquist (MATLAB HalfPowerFrequency)
STD_BLOCK    = 100        # samples per block for the minimum-std noise estimate
MAX_POINTS   = 50_000     # points stored per time trace in the plots (keeps the web page light)

available = sorted(p.name for p in DATA_DIR.iterdir() if p.is_file() and lvm.is_lvm(p))
print(f"{len(available)} LVM files in {DATA_DIR}:", *available, sep="\n  ")

15 LVM files in ..\05luglio:
  AB.lvm
  AB_1.lvm
  AB_bis.lvm
  AB_bis_1
  AB_bis_1.lvm
  PEG_1
  baselineDIW_1
  baselineDIW_ferma
  baselinePBS_1
  spike01
  spike01.lvm
  spike1
  spike1.lvm
  spike10
  spike10.lvm


## 1. Single file

In [3]:
d = lvm.read_lvm(DATA_DIR / SINGLE_FILE)
print(d)
print("Recorded:", d.header.get("Date"), d.header.get("Time"), "| operator:", d.header.get("Operator"))
d.data.describe().T

LVMData('AB.lvm', channels=['Phase1', 'Phase2', 'ModulatingSignal', 'PD'], samples=120000, segments=6, fs=1000 Hz)
Recorded: 2021/07/05 14:22:30.0145998001098632813 | operator: FOS-RGD


,count,mean,std,min,25%,50%,75%,max
time,120000.0,59.999500,34.641160,0.000000,29.999750,59.999500,89.999250,119.999000
Phase1,119000.0,-14851.189093,38925.064175,-117855.192867,-15.193168,-14.131887,-13.163600,45.862994
Phase2,119000.0,-1506.881058,4090.932211,-12337.457367,53.756689,54.688455,55.288539,57.430839
ModulatingSignal,120000.0,0.011091,2.669430,-3.791104,-2.646448,0.011596,2.668650,3.811246
PD,120000.0,0.619243,0.054452,0.460975,0.568093,0.633707,0.660563,0.765850


### Time traces

The phase channels come first, with the photodiode (PD) on its own axis below because its units differ. Both panels share the time axis, so zooming one zooms the other.

In [4]:
df = d.data
if T_WINDOW is not None:
    df = df[(df.time >= T_WINDOW[0]) & (df.time <= T_WINDOW[1])]
t = df.time.values

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.72, 0.28], vertical_spacing=0.04)
for i, ch in enumerate(["Phase1", "Phase2"]):
    y = lvm.unwrap_phase(df[ch].values) if UNWRAP else df[ch].values
    xs, ys = lvm.decimate_for_plot(t, y, MAX_POINTS)
    fig.add_trace(go.Scattergl(x=xs, y=ys, name=ch, line=dict(color=SERIES[i], width=1.2)), row=1, col=1)
xs, ys = lvm.decimate_for_plot(t, df.PD.values, MAX_POINTS)
fig.add_trace(go.Scattergl(x=xs, y=ys, name="PD", line=dict(color=SERIES[2], width=1)), row=2, col=1)
fig.update_yaxes(title_text="phase [°]", row=1, col=1)
fig.update_yaxes(title_text="PD [V]", row=2, col=1)
fig.update_xaxes(title_text="time [s]", row=2, col=1)
fig.update_layout(title=SINGLE_FILE, height=600)
fig.show(config=PLOT_CONFIG)

### Power spectral density

One-sided FFT periodogram, PSD = 2|X(f)|² / (f<sub>s</sub> N), computed exactly as in the MATLAB script. NaN samples, which occur while the PGC demodulation locks, are dropped. The faint line is the raw periodogram and the solid line is its average over log-spaced frequency bins.

In [5]:
fig = go.Figure()
for i, ch in enumerate(["Phase1", "Phase2"]):
    y = df[ch].dropna().values
    f, p = lvm.periodogram(y, d.fs)
    xs, ys = lvm.decimate_for_plot(f[1:], 10*np.log10(p[1:]), 20_000)
    fig.add_trace(go.Scattergl(x=xs, y=ys, name=f"{ch} raw", legendgroup=ch, showlegend=False,
                               opacity=0.25, line=dict(color=SERIES[i], width=0.6), hoverinfo="skip"))
    fb, pb = lvm.log_bin(f, p)
    fig.add_trace(go.Scatter(x=fb, y=10*np.log10(pb), name=ch, legendgroup=ch, line=dict(color=SERIES[i], width=2)))
fig.update_xaxes(type="log", title_text="frequency [Hz]")
fig.update_yaxes(title_text="PSD [dB re 1 °²/Hz]")
fig.update_layout(title=f"Periodogram using FFT: {SINGLE_FILE}")
fig.show(config=PLOT_CONFIG)

## 2. Multiple files

For each file, on the `PHASE_CH` channel:
* NaNs are removed;
* the slow drift is estimated with a 12th-order zero-phase Butterworth low-pass (`designfilt` + `filtfilt` in MATLAB) and subtracted before the PSD is taken;
* the **mean** phase and the **minimum standard deviation** over consecutive `STD_BLOCK`-sample blocks are recorded. The minimum std is a measure of the noise floor.

A decimated copy of each trace is kept for the chronogram.

In [6]:
rows, spectra, traces = [], [], []
for name, label in zip(MULTI_FILES, LABELS):
    di = lvm.read_lvm(DATA_DIR / name)
    sub = di.data[["time", PHASE_CH]].dropna()
    t, y = sub.time.values, sub[PHASE_CH].values
    if UNWRAP:
        y = lvm.unwrap_phase(y)
    trend = lvm.lowpass_trend(y, TREND_CUTOFF)
    f, p = lvm.periodogram(y - trend, di.fs)
    spectra.append((label, *lvm.log_bin(f, p)))
    traces.append((label, *lvm.decimate_for_plot(t, y, CHRONO_POINTS)))
    rows.append(dict(file=name, label=label, fs_Hz=di.fs, duration_s=len(y)/di.fs,
                     segments=di.n_segments, mean_deg=y.mean(),
                     min_block_std_deg=lvm.min_block_std(y, STD_BLOCK)))
    del di, sub
summary = pd.DataFrame(rows).set_index("label")
summary

,file,fs_Hz,duration_s,segments,mean_deg,min_block_std_deg
label,,,,,,
DIW,baselineDIW_1,1000.0,199.0,10,-21523.683600,0.428214
PBS,baselinePBS_1,1000.0,179.0,9,-774.374565,0.530458
PEG,PEG_1,1000.0,4639.0,232,4938.662752,0.353591
AB,AB.lvm,1000.0,119.0,6,-14851.189093,0.440663
0.1 nM,spike01.lvm,1000.0,3399.0,170,190.362074,0.522396
1 nM,spike1.lvm,1000.0,3699.0,185,-11948.388384,0.216441
10 nM,spike10.lvm,1000.0,3539.0,177,-2906.309360,0.182833


### Chronogram

The files are placed end to end in time. As in `analysistrace05Luglio.m`, each trace is shifted so that it starts where the previous one ended. This hides the arbitrary phase offset between recordings. Set `ALIGN = False` to plot the raw phase values.

In [7]:
ALIGN = True
fig = go.Figure()
t0, y_end = 0.0, 0.0
for i, (label, t, y) in enumerate(traces):
    shift = (y_end - y[0]) if ALIGN else 0.0
    fig.add_trace(go.Scattergl(x=t + t0, y=y + shift, name=label,
                               line=dict(color=SERIES[i % len(SERIES)], width=1.2)))
    t0, y_end = t0 + t[-1], y[-1] + shift
fig.update_xaxes(title_text="time [s]")
fig.update_yaxes(title_text=f"{PHASE_CH} [°]" + (" (aligned)" if ALIGN else ""))
fig.update_layout(title=f"Chronogram: {PHASE_CH}", height=550)
fig.show(config=PLOT_CONFIG)

### Detrended phase PSD

In [8]:
fig = go.Figure()
for i, (label, fb, pb) in enumerate(spectra):
    fig.add_trace(go.Scatter(x=fb, y=10*np.log10(pb), name=label,
                             line=dict(color=SERIES[i % len(SERIES)], width=1.5)))
fig.update_xaxes(type="log", title_text="frequency [Hz]")
fig.update_yaxes(title_text="PSD [dB re 1 °²/Hz]")
fig.update_layout(title=f"Detrended {PHASE_CH} PSD (log-binned)")
fig.show(config=PLOT_CONFIG)

### Mean phase and noise floor per file

The MATLAB script plotted these against a `Temperature` array whose parsing from the file name was commented out. Here the files are on the x-axis in the order of `MULTI_FILES`. To plot against a physical quantity, add a column to `summary` (concentration, temperature, …) and use it as `x`.

In [9]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=["Mean ± min block std", f"Noise floor ({STD_BLOCK}-sample blocks)"])
marker = dict(size=10, color=SERIES[0], line=dict(color="#fcfcfb", width=2))
fig.add_trace(go.Scatter(x=summary.index, y=summary.mean_deg, mode="markers", marker=marker,
                         error_y=dict(array=summary.min_block_std_deg, color=SERIES[0]), name="mean"), 1, 1)
fig.add_trace(go.Scatter(x=summary.index, y=summary.min_block_std_deg, mode="lines+markers",
                         marker=marker, line=dict(color=SERIES[0], width=2), name="min std"), 1, 2)
fig.update_yaxes(title_text=f"mean {PHASE_CH} [°]", row=1, col=1)
fig.update_yaxes(title_text="min block std [°]", row=1, col=2)
fig.update_layout(showlegend=False, hovermode="closest", height=420)
fig.show(config=PLOT_CONFIG)

## Differences from `LVM_reader.m`

* **Time axis across segments**: MATLAB offset each segment by the previous segment's last timestamp, which repeats one timestamp. Here the offset is `last + dt`.
* **Sampling rate**: read from the file's `Delta_X` header and no longer parsed from the file name. `lvm.fs_from_filename()` still provides the old `…kHz` rule.
* **Unwrap**: `unwrap` in MATLAB assumes radians, but the phase is in degrees and already unwrapped. Unwrapping is therefore off by default. `lvm.unwrap_phase()` uses a 360° period when enabled.
* **Low-pass filter**: implemented with second-order sections (`sosfiltfilt`). A 12th-order Butterworth filter at 1e-4 is numerically unstable in transfer-function form.
* **Single-file PSD**: shown in Hz and dB/Hz rather than normalised frequency.